In [1]:
import pandas as pd
import numpy as np

In [3]:
# =====================================================================
# PHASE 1: GENERATE THE CHAOTIC DATASET (Simulating messy MR Field Reports)
# =====================================================================
print("📦 Phase 1: Simulating Chaotic Field Datasets...")
raw_data = {
    'Date': ['2026-01-01', '2026/01/02', '03-01-2026', '2026-01-04', '2026-01-05', '2026-01-06'],
    'Region': ['South', None, 'West', 'East', 'North', 'South'],
    'Product_Name': ['Paracetamol', 'Amoxicillin', 'Unknown', 'Vitamin-C', 'B-Complex', 'Paracetamol'],
    'Units_Sold': [20, 10, 50, -5, 9999, None],        # Contains a negative, a massive typo, and a missing value
    'Price_Per_Unit_INR': [50, 120, None, 80, 45, 50]   # Missing price for the 'Unknown' product
}

📦 Phase 1: Simulating Chaotic Field Datasets...


In [6]:
raw_data

{'Date': ['2026-01-01',
  '2026/01/02',
  '03-01-2026',
  '2026-01-04',
  '2026-01-05',
  '2026-01-06'],
 'Region': ['South', None, 'West', 'East', 'North', 'South'],
 'Product_Name': ['Paracetamol',
  'Amoxicillin',
  'Unknown',
  'Vitamin-C',
  'B-Complex',
  'Paracetamol'],
 'Units_Sold': [20, 10, 50, -5, 9999, None],
 'Price_Per_Unit_INR': [50, 120, None, 80, 45, 50]}

In [7]:
df=pd.DataFrame(raw_data)
print("\n--- Original Chaotic Data ---")
print(df)


--- Original Chaotic Data ---
         Date Region Product_Name  Units_Sold  Price_Per_Unit_INR
0  2026-01-01  South  Paracetamol        20.0                50.0
1  2026/01/02   None  Amoxicillin        10.0               120.0
2  03-01-2026   West      Unknown        50.0                 NaN
3  2026-01-04   East    Vitamin-C        -5.0                80.0
4  2026-01-05  North    B-Complex      9999.0                45.0
5  2026-01-06  South  Paracetamol         NaN                50.0


In [8]:
# =====================================================================
# PHASE 2: DATA CLEANING & RESTRUCTURING PIPELINE
# =====================================================================
print("\n⚙️ Phase 2: Running the Pandas & NumPy Cleaning Pipeline...")


⚙️ Phase 2: Running the Pandas & NumPy Cleaning Pipeline...


In [52]:
df['Date']=pd.to_datetime(df['Date'],format='mixed',errors='coerce')
print(df['Date'])

0   2026-01-01
1          NaT
2          NaT
3   2026-01-04
4   2026-01-05
5   2026-01-06
Name: Date, dtype: datetime64[ns]


In [24]:
# 2. Fill Missing (Null) Values using Pandas
# If Region is missing, flag it as 'Unassigned'
df['Region'] = df['Region'].fillna('Unassigned')

In [26]:
df

,Date,Region,Product_Name,Units_Sold,Price_Per_Unit_INR
0,2026-01-01,South,Paracetamol,20.0,50.0
1,NaT,Unassigned,Amoxicillin,10.0,120.0
2,NaT,West,Unknown,50.0,NaN
3,2026-01-04,East,Vitamin-C,-5.0,80.0
4,2026-01-05,North,B-Complex,9999.0,45.0
5,2026-01-06,South,Paracetamol,NaN,50.0


In [33]:
# 3. Clean Out-of-Bound Variables using NumPy
# A: Units_Sold cannot be negative. If it's less than 0, flip it to positive or make it 0.
df['Units_Sold'] = np.where(df['Units_Sold'] < 0 , 0, df['Units_Sold'])

In [35]:
df['Units_Sold']

,Units_Sold
0,20.0
1,10.0
2,50.0
3,0.0
4,9999.0
5,NaN


In [37]:
# B: 9999 is clearly a typo/outlier. If Units_Sold > 500, replace it with the column median.
# We calculate the median ignoring the 9999 anomaly first.
valid_units_median = df[df['Units_Sold'] < 500]['Units_Sold'].median()
df['Units_Sold'] = np.where(df['Units_Sold'] > 500, valid_units_median, df['Units_Sold'])

In [40]:
# C: Replace remaining missing (NaN) values in Units_Sold with the clean median
df['Units_Sold'] = df['Units_Sold'].fillna(valid_units_median)

In [42]:
df['Units_Sold']

,Units_Sold
0,20.0
1,10.0
2,50.0
3,0.0
4,15.0
5,15.0


In [46]:
df['Total_Revenue_INR'] = df['Units_Sold'] * df['Price_Per_Unit_INR']


In [49]:
df

,Date,Region,Product_Name,Units_Sold,Price_Per_Unit_INR,Total_Revenue_INR
0,2026-01-01,South,Paracetamol,20.0,50.0,1000.0
1,NaT,Unassigned,Amoxicillin,10.0,120.0,1200.0
2,NaT,West,Unknown,50.0,NaN,NaN
3,2026-01-04,East,Vitamin-C,0.0,80.0,0.0
4,2026-01-05,North,B-Complex,15.0,45.0,675.0
5,2026-01-06,South,Paracetamol,15.0,50.0,750.0


In [54]:
# =====================================================================
# PHASE 3: AGGREGATION FOR KAPIVA EXECUTIVES (Insight Generation)
# =====================================================================
print("\n📊 Phase 3: Generating Executive Aggregations...")

# Group by Region to track performance metrics
regional_summary = df.groupby('Region').agg(
    Total_Quantity=('Units_Sold', 'sum'),
    Total_Sales_INR=('Total_Revenue_INR', 'sum')
).reset_index()

print("\n--- Final Cleaned Regional Sales Dashboard Input ---")
print(regional_summary)


📊 Phase 3: Generating Executive Aggregations...

--- Final Cleaned Regional Sales Dashboard Input ---
       Region  Total_Quantity  Total_Sales_INR
0        East             0.0              0.0
1       North            15.0            675.0
2       South            35.0           1750.0
3  Unassigned            10.0           1200.0
4        West            50.0              0.0


In [53]:
print("\n--- Cleaned & Restructured Data ---")
print(df)


--- Cleaned & Restructured Data ---
        Date      Region Product_Name  Units_Sold  Price_Per_Unit_INR  \
0 2026-01-01       South  Paracetamol        20.0                50.0   
1        NaT  Unassigned  Amoxicillin        10.0               120.0   
2        NaT        West      Unknown        50.0                 NaN   
3 2026-01-04        East    Vitamin-C         0.0                80.0   
4 2026-01-05       North    B-Complex        15.0                45.0   
5 2026-01-06       South  Paracetamol        15.0                50.0   

   Total_Revenue_INR  
0             1000.0  
1             1200.0  
2                NaN  
3                0.0  
4              675.0  
5              750.0  
